In [1]:
pip install decord transformers accelerate timm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 91.5 MB/s eta 0:00:00:00:010:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from pathlib import Path
import os
import torch
import torch.nn as nn
import timm
import random
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR
from decord import VideoReader, cpu
from tqdm import tqdm


VIDEO_ROOT = Path("/kaggle/input/datasets/indiff/videos")  
CSV_PATH = Path("/kaggle/input/datasets/indiff/labels/bah-video.csv")


df = pd.read_csv(CSV_PATH)


df["full_path"] = df["video-path"].apply(
    lambda x: str(VIDEO_ROOT / str(x).strip())
)

print("Проверка путей:\n")

for p in df["full_path"].sample(5):
    print(p)
    print("Exists:", os.path.exists(p))
    print("-" * 60)


/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

Проверка путей:

/kaggle/input/datasets/indiff/videos/Videos/82981/Visite_1/82981_Question_1_2025-04-28_12-44-15_Video.mp4
Exists: True
------------------------------------------------------------
/kaggle/input/datasets/indiff/videos/Videos/82815/Visite_1/82815_Question_7_2025-01-31_22-22-29_Video.mp4
Exists: True
------------------------------------------------------------
/kaggle/input/datasets/indiff/videos/Videos/82825/Visite_1/82825_Question_3_2025-02-05_18-38-39_Video.mp4
Exists: True
------------------------------------------------------------
/kaggle/input/datasets/indiff/videos/Videos/83045/Visite_1/83045_Question_1_2025-05-08_12-45-13_Video.mp4
Exists: True
------------------------------------------------------------
/kaggle/input/datasets/indiff/videos/Videos/83088/Visite_1/83088_Question_1_2025-05-29_12-59-52_Video.mp4
Exists: True
------------------------------------------------------------


In [3]:
(df['label'] == 0).sum()

np.int64(649)

In [4]:
print(df["video-path"].iloc[0])
print(df["full_path"].iloc[0])


Videos/82694/Visite_1/82694_Question_1_2024-11-15_21-05-54_Video.mp4
/kaggle/input/datasets/indiff/videos/Videos/82694/Visite_1/82694_Question_1_2024-11-15_21-05-54_Video.mp4


In [5]:
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=42)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Размер выборок -> Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Размер выборок -> Train: 998 | Val: 214 | Test: 215


In [15]:
BATCH_SIZE = 4
NUM_FRAMES = 16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

In [16]:
# Аугментации
train_transform = v2.Compose([
    v2.Resize((256, 256), antialias=True),
    v2.RandomResizedCrop(224, scale=(0.8, 1.0), antialias=True),
    v2.RandomHorizontalFlip(p=0.5),
    v2.ColorJitter(brightness=0.2, contrast=0.2),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = v2.Compose([
    v2.Resize((224, 224), antialias=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class RobustVideoDataset(Dataset):
    def __init__(self, dataframe, num_frames=16, transform=None, is_train=False):
        self.df = dataframe
        self.num_frames = num_frames
        self.transform = transform
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def load_video(self, path):
        vr = VideoReader(path, ctx=cpu(0))
        total_frames = len(vr)
        
        if total_frames < self.num_frames:
            # Если видео слишком короткое, дублируем кадры
            indices = np.linspace(0, total_frames - 1, self.num_frames).astype(int)
        elif self.is_train:
            # Временной джиттеринг для аугментации
            max_offset = (total_frames - 1) // self.num_frames
            start = random.randint(0, max_offset) if max_offset > 0 else 0
            indices = np.linspace(start, total_frames - 1, self.num_frames).astype(int)
        else:
            indices = np.linspace(0, total_frames - 1, self.num_frames).astype(int)

        frames = vr.get_batch(indices).asnumpy()
        frames = torch.from_numpy(frames).permute(0, 3, 1, 2).float() / 255.0
        return frames

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        video = self.load_video(row["full_path"])
        if self.transform:
            video = self.transform(video)
        label = torch.tensor(row["label"]).long()
        return video, label

train_loader = DataLoader(RobustVideoDataset(train_df, NUM_FRAMES, train_transform, is_train=True), 
                          batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(RobustVideoDataset(val_df, NUM_FRAMES, val_test_transform, is_train=False), 
                        batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader = DataLoader(RobustVideoDataset(test_df, NUM_FRAMES, val_test_transform, is_train=False), 
                         batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [10]:
from transformers import VideoMAEForVideoClassification
model = VideoMAEForVideoClassification.from_pretrained(
    "MCG-NJU/videomae-base-finetuned-kinetics", 
    num_labels=2,
    ignore_mismatched_sizes=True
)

model.to(device)

2026-02-25 10:17:14.072171: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772014634.263732      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772014634.322380      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772014634.752800      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772014634.752852      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772014634.752855      55 computation_placer.cc:177] computation placer alr

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of VideoMAEForVideoClassification were not initialized from the model checkpoint at MCG-NJU/videomae-base-finetuned-kinetics and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([400]) in the checkpoint and torch.Size([2]) in the model instantiated
- classifier.weight: found shape torch.Size([400, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


VideoMAEForVideoClassification(
  (videomae): VideoMAEModel(
    (embeddings): VideoMAEEmbeddings(
      (patch_embeddings): VideoMAEPatchEmbeddings(
        (projection): Conv3d(3, 768, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
    )
    (encoder): VideoMAEEncoder(
      (layer): ModuleList(
        (0-11): 12 x VideoMAELayer(
          (attention): VideoMAEAttention(
            (attention): VideoMAESelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=False)
              (key): Linear(in_features=768, out_features=768, bias=False)
              (value): Linear(in_features=768, out_features=768, bias=False)
            )
            (output): VideoMAESelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): VideoMAEIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
    

In [11]:
total_samples = 778 + 649
weight_0 = total_samples / (2.0 * 649)
weight_1 = total_samples / (2.0 * 778)
class_weights = torch.tensor([weight_0, weight_1]).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

optimizer_grouped_parameters = [
    {"params": model.videomae.parameters(), "lr": 1e-5}, 
    {"params": model.classifier.parameters(), "lr": 5e-4}
]
optimizer = torch.optim.AdamW(optimizer_grouped_parameters, weight_decay=0.05)

ACCUMULATION_STEPS = 4 
EPOCHS = 20

In [12]:
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=[1e-5, 5e-4], # Максимальные LR для каждой группы
    steps_per_epoch=len(train_loader) // ACCUMULATION_STEPS + 1,
    epochs=EPOCHS,
    pct_start=0.1 # Первые 10% времени LR будет плавно расти
)

In [13]:
def train_epoch():
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    
    for i, (videos, labels) in enumerate(tqdm(train_loader, desc="Training")):
        videos, labels = videos.to(device), labels.to(device)
        
        outputs = model(videos).logits
        loss = criterion(outputs, labels)
        
        loss = loss / ACCUMULATION_STEPS
        loss.backward()
        
        if (i + 1) % ACCUMULATION_STEPS == 0 or (i + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            
        total_loss += loss.item() * ACCUMULATION_STEPS # Возвращаем реальный масштаб лосса
        
    return total_loss / len(train_loader)

def evaluate(loader, desc="Evaluating"):
    model.eval()
    preds, true = [], []
    with torch.no_grad():
        for videos, labels in tqdm(loader, desc=desc):
            videos, labels = videos.to(device), labels.to(device)
            outputs = model(videos).logits
            predicted = torch.argmax(outputs, dim=1)
            
            preds.extend(predicted.cpu().numpy())
            true.extend(labels.cpu().numpy())
            
    acc = accuracy_score(true, preds)
    mf1 = f1_score(true, preds, average='macro')
    return acc, mf1

In [17]:
best_val_mf1 = 0.0

for epoch in range(EPOCHS):
    print(f"\n--- Epoch {epoch+1}/{EPOCHS} ---")
    loss = train_epoch()
    val_acc, val_mf1 = evaluate(val_loader, desc="Validation")
    
    print(f"Loss: {loss:.4f} | Val Acc: {val_acc:.4f} | Val Macro F1: {val_mf1:.4f}")
    
    if val_mf1 > best_val_mf1:
        best_val_mf1 = val_mf1
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"-> Модель улучшилась (New Best MF1: {best_val_mf1:.4f}), сохраняем!")


--- Epoch 1/20 ---


Validation: 100%|██████████| 54/54 [01:31<00:00,  1.69s/it]


Loss: 0.7020 | Val Acc: 0.4720 | Val Macro F1: 0.3886
-> Модель улучшилась (New Best MF1: 0.3886), сохраняем!

--- Epoch 2/20 ---


Validation: 100%|██████████| 54/54 [01:31<00:00,  1.69s/it]


Loss: 0.6810 | Val Acc: 0.5935 | Val Macro F1: 0.5624
-> Модель улучшилась (New Best MF1: 0.5624), сохраняем!

--- Epoch 3/20 ---


...

Loss: 0.3321 | Val Acc: 0.5935 | Val Macro F1: 0.5869

--- Epoch 19/20 ---


Validation: 100%|██████████| 54/54 [01:29<00:00,  1.65s/it]


Loss: 0.3274 | Val Acc: 0.5888 | Val Macro F1: 0.5816

--- Epoch 20/20 ---


Validation: 100%|██████████| 54/54 [01:28<00:00,  1.65s/it]

Loss: 0.3272 | Val Acc: 0.5888 | Val Macro F1: 0.5816


In [19]:
model.load_state_dict(torch.load('best_model.pth'))
test_acc, test_mf1 = evaluate(test_loader, desc="Testing")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Macro F1: {test_mf1:.4f}")

Testing: 100%|██████████| 54/54 [01:25<00:00,  1.58s/it]

Test Accuracy: 0.5953
Test Macro F1: 0.5950
